# SL-Cook100 Dataset — Jupyter Notebook
**BSc Computer Science & Data Science — Project Dataset Analysis**

This notebook loads, validates, and analyses the SL-Cook100 Sri Lankan recipe dataset.

---
## Setup: Point to your dataset folder

In [ ]:
import os
import json
import glob
from collections import defaultdict, Counter

# ── CHANGE THIS to wherever you saved your dataset files ──
DATASET_DIR = './slcook100'   # e.g. 'C:/Users/Fawaaz/Downloads/slcook100'

TAXONOMY_FILE   = os.path.join(DATASET_DIR, 'ingredient_taxonomy.json')
VOCAB_FILE      = os.path.join(DATASET_DIR, 'tag_vocabulary.json')
RECIPE_PATTERN  = os.path.join(DATASET_DIR, 'sl_*.json')

print('Dataset directory:', os.path.abspath(DATASET_DIR))
print('Recipe files found:', len(glob.glob(RECIPE_PATTERN)))

## 1. Load the dataset

In [ ]:
# Load taxonomy and tag vocabulary
with open(TAXONOMY_FILE, encoding='utf-8') as f:
    taxonomy = json.load(f)
with open(VOCAB_FILE, encoding='utf-8') as f:
    vocab = json.load(f)

# Load all recipe files
recipes = []
for filepath in sorted(glob.glob(RECIPE_PATTERN)):
    with open(filepath, encoding='utf-8') as f:
        recipes.append(json.load(f))

# Build lookup sets
tax_ids     = {i['id'] for i in taxonomy['ingredients']}
valid_tags  = {t for cat in vocab['categories'].values() for t in cat['tags']}

print(f'Recipes loaded:              {len(recipes)}')
print(f'Canonical ingredient IDs:    {len(tax_ids)}')
print(f'Valid vocabulary tags:        {len(valid_tags)}')

## 2. Full Validation

In [ ]:
REQUIRED_FIELDS = [
    'id', 'name_en', 'name_si', 'regional_origin', 'cuisine', 'course',
    'servings', 'total_time_min', 'steps', 'ingredients', 'tags',
    'ayurvedic_balance', 'nutrition_per_serving', 'trust_score', 'source_url'
]

errors   = []
warnings = []
all_ids  = []

for r in recipes:
    rid = r.get('id', 'UNKNOWN')
    all_ids.append(rid)

    # 1. Required fields
    for field in REQUIRED_FIELDS:
        if field not in r:
            errors.append(f'{rid}: MISSING FIELD "{field}"')

    # 2. Canonical ingredient IDs
    for ing in r.get('ingredients', []):
        if ing['canonical_id'] not in tax_ids:
            errors.append(f'{rid}: Unknown canonical_id "{ing["canonical_id"]}"')

    # 3. Tags in controlled vocabulary
    for tag in r.get('tags', []):
        if tag not in valid_tags:
            errors.append(f'{rid}: Unknown tag "{tag}"')

    # 4. Trust score range
    ts = r.get('trust_score', 0)
    if not (0 <= ts <= 1):
        errors.append(f'{rid}: trust_score {ts} out of range [0,1]')

    # 5. Steps have duration_min
    for step in r.get('steps', []):
        if 'duration_min' not in step:
            warnings.append(f'{rid} step {step.get("step")}: missing duration_min')

# 6. Duplicate IDs
dup_ids = [id_ for id_, count in Counter(all_ids).items() if count > 1]
if dup_ids:
    errors.append(f'DUPLICATE IDs found: {dup_ids}')

# Report
print('=== VALIDATION REPORT ===')
print(f'Recipes checked:  {len(recipes)}')
print(f'Errors:           {len(errors)}')
print(f'Warnings:         {len(warnings)}')
print()
if errors:
    print('ERRORS:')
    for e in errors:
        print(f'  ✗ {e}')
else:
    print('✓ No errors — all 99 recipes pass validation')
if warnings:
    print('\nWARNINGS:')
    for w in warnings:
        print(f'  ⚠ {w}')

## 3. Dataset Statistics

In [ ]:
import statistics

trust_scores    = [r['trust_score'] for r in recipes]
total_times     = [r.get('total_time_min', 0) for r in recipes]
step_counts     = [len(r.get('steps', [])) for r in recipes]
ing_counts      = [len(r.get('ingredients', [])) for r in recipes]
servings_all    = [r.get('servings', 0) for r in recipes]

print('=== DATASET STATISTICS ===')
print(f'Total recipes:              {len(recipes)}')
print(f'Canonical ingredients:      {len(tax_ids)}')
print(f'Controlled vocabulary tags: {len(valid_tags)}')
print()
print('Trust Scores:')
print(f'  Mean:    {statistics.mean(trust_scores):.3f}')
print(f'  Median:  {statistics.median(trust_scores):.3f}')
print(f'  Min:     {min(trust_scores):.2f}  ({[r["id"] for r in recipes if r["trust_score"]==min(trust_scores)]})')
print(f'  Max:     {max(trust_scores):.2f}  ({[r["id"] for r in recipes if r["trust_score"]==max(trust_scores)]})')
print()
print('Ingredients per recipe:')
print(f'  Mean:  {statistics.mean(ing_counts):.1f}  |  Min: {min(ing_counts)}  |  Max: {max(ing_counts)}')
print()
print('Steps per recipe:')
print(f'  Mean:  {statistics.mean(step_counts):.1f}  |  Min: {min(step_counts)}  |  Max: {max(step_counts)}')
print()
print('Total time (wall-clock minutes):')
print(f'  Mean:   {statistics.mean(total_times):.0f} min')
print(f'  Median: {statistics.median(total_times):.0f} min')
print(f'  Fastest: {min(total_times)} min  |  Slowest: {max(total_times)} min')

## 4. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('SL-Cook100 Dataset Analysis', fontsize=15, fontweight='bold', y=1.01)

# 1. Recipes by course
course_counts = Counter(r['course'] for r in recipes)
axes[0,0].barh(list(course_counts.keys()), list(course_counts.values()), color='steelblue')
axes[0,0].set_title('Recipes by Course')
axes[0,0].set_xlabel('Count')

# 2. Trust score distribution
axes[0,1].hist(trust_scores, bins=15, color='darkorange', edgecolor='white')
axes[0,1].axvline(statistics.mean(trust_scores), color='red', linestyle='--', label=f'Mean: {statistics.mean(trust_scores):.3f}')
axes[0,1].set_title('Trust Score Distribution')
axes[0,1].set_xlabel('Trust Score')
axes[0,1].set_ylabel('Number of Recipes')
axes[0,1].legend()

# 3. Top 15 tags
tag_counter = Counter(t for r in recipes for t in r.get('tags', []))
top_tags = tag_counter.most_common(15)
axes[0,2].barh([t[0] for t in top_tags], [t[1] for t in top_tags], color='mediumseagreen')
axes[0,2].set_title('Top 15 Tags')
axes[0,2].set_xlabel('Number of Recipes')
axes[0,2].invert_yaxis()

# 4. Ingredient count distribution
axes[1,0].hist(ing_counts, bins=range(min(ing_counts), max(ing_counts)+2), color='mediumpurple', edgecolor='white')
axes[1,0].set_title('Ingredients per Recipe')
axes[1,0].set_xlabel('Number of Ingredients')
axes[1,0].set_ylabel('Number of Recipes')

# 5. Total time distribution (capped at 200 min for readability)
capped_times = [min(t, 200) for t in total_times]
axes[1,1].hist(capped_times, bins=20, color='tomato', edgecolor='white')
axes[1,1].set_title('Total Time Distribution\n(capped at 200 min for readability)')
axes[1,1].set_xlabel('Total Time (minutes)')
axes[1,1].set_ylabel('Number of Recipes')

# 6. Ayurvedic balance pie chart
ayur_counts = Counter(r.get('ayurvedic_balance', 'unknown') for r in recipes)
axes[1,2].pie(ayur_counts.values(), labels=ayur_counts.keys(), autopct='%1.0f%%',
              colors=['#f4a460','#87ceeb','#90ee90','#dda0dd'])
axes[1,2].set_title('Ayurvedic Balance Distribution')

plt.tight_layout()
plt.savefig(os.path.join(DATASET_DIR, 'slcook100_analysis.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Plot saved to slcook100_analysis.png')

## 5. Community & Cultural Coverage

In [ ]:
# Regional origin analysis
regional = Counter(r.get('regional_origin', '') for r in recipes)
print('=== REGIONAL ORIGIN ANALYSIS ===')
for region, count in regional.most_common():
    print(f'  {count:2d}  {region}')

# Community clusters based on keywords in regional_origin
communities = {
    'Island-wide': [r for r in recipes if 'island-wide' in r.get('regional_origin','').lower()],
    'Colombo / Fusion': [r for r in recipes if 'colombo' in r.get('regional_origin','').lower()],
    'Tamil (Northern/Eastern)': [r for r in recipes if 'northern' in r.get('regional_origin','').lower() or 'eastern' in r.get('regional_origin','').lower() or 'jaffna' in r.get('regional_origin','').lower() or 'tamil' in r.get('regional_origin','').lower()],
    'Coastal': [r for r in recipes if 'coastal' in r.get('regional_origin','').lower()],
    'Burgher / Colonial': [r for r in recipes if 'burgher' in r.get('regional_origin','').lower() or 'dutch' in r.get('regional_origin','').lower() or 'portuguese' in r.get('regional_origin','').lower()],
    'Muslim / Malay': [r for r in recipes if 'muslim' in r.get('regional_origin','').lower() or 'malay' in r.get('regional_origin','').lower()],
    'Upcountry': [r for r in recipes if 'upcountry' in r.get('regional_origin','').lower() or 'kandy' in r.get('regional_origin','').lower() or 'nuwara' in r.get('regional_origin','').lower()],
}

print('\n=== COMMUNITY / CULTURAL COVERAGE ===')
for community, recs in communities.items():
    print(f'  {community}: {len(recs)} recipes')
    for r in recs:
        print(f'    - {r["id"]}: {r["name_en"]}')

## 6. Ingredient Taxonomy Analysis

In [ ]:
# Which ingredients appear most often across recipes
ing_usage = Counter()
for r in recipes:
    seen = set()
    for ing in r.get('ingredients', []):
        cid = ing['canonical_id']
        if cid not in seen:  # count each ingredient once per recipe
            ing_usage[cid] += 1
            seen.add(cid)

print('=== TOP 20 MOST USED CANONICAL INGREDIENTS ===')
tax_lookup = {i['id']: i['name'] for i in taxonomy['ingredients']}
for cid, count in ing_usage.most_common(20):
    print(f'  {count:3d} recipes  {cid:30s}  {tax_lookup.get(cid, "?")}')

print()
# Ingredients used in only 1 recipe
rare_ings = [(cid, tax_lookup.get(cid,'?')) for cid, count in ing_usage.items() if count == 1]
print(f'Ingredients used in only 1 recipe: {len(rare_ings)}')
for cid, name in rare_ings:
    print(f'  {cid}: {name}')

# Defined in taxonomy but never used
never_used = tax_ids - set(ing_usage.keys())
print(f'\nIngredients defined in taxonomy but not yet used: {len(never_used)}')
for cid in sorted(never_used):
    print(f'  {cid}: {tax_lookup.get(cid,"?")}')

## 7. Cross-Recipe Similarity (Shared Ingredients)

In [ ]:
# Find recipe pairs that share the most canonical ingredients
ing_sets = {r['id']: set(i['canonical_id'] for i in r.get('ingredients',[])) for r in recipes}

pairs = []
ids = list(ing_sets.keys())
for i in range(len(ids)):
    for j in range(i+1, len(ids)):
        shared = ing_sets[ids[i]] & ing_sets[ids[j]]
        if len(shared) >= 8:  # only show pairs sharing 8+ ingredients
            pairs.append((len(shared), ids[i], ids[j], shared))

pairs.sort(reverse=True)
print('=== RECIPE PAIRS WITH 8+ SHARED CANONICAL INGREDIENTS ===')
name_lookup = {r['id']: r['name_en'] for r in recipes}
for count, id1, id2, shared in pairs[:15]:
    print(f'  {count} shared: {id1} ({name_lookup[id1]}) <-> {id2} ({name_lookup[id2]})')
    print(f'    Shared: {", ".join(sorted(shared))}')

## 8. Export Combined Dataset JSON (optional)

In [ ]:
# Combine all recipes into a single JSON array (useful for model training)
combined = {
    'dataset_name': 'SL-Cook100',
    'version': '1.0',
    'description': 'A structured dataset of 99 Sri Lankan recipes (sl_002 to sl_100) with ingredients, steps, tags, nutrition, and Ayurvedic annotations.',
    'total_recipes': len(recipes),
    'recipes': sorted(recipes, key=lambda r: r['id'])
}

out_path = os.path.join(DATASET_DIR, 'slcook100_combined.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(combined, f, indent=2, ensure_ascii=False)

size_kb = os.path.getsize(out_path) / 1024
print(f'Combined dataset written to: {out_path}')
print(f'File size: {size_kb:.1f} KB ({size_kb/1024:.2f} MB)')
print(f'Total recipes in combined file: {len(combined["recipes"])}')